In [2]:
import numpy as np
import tensorflow as tf
import os
import shutil
import json
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
tf.config.list_physical_devices("GPU")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [9]:
DATA_DIR = os.path.join(os.path.dirname(os.path.dirname(os.environ.get("JPY_SESSION_NAME"))), "data")
IM_DIR = os.path.join(DATA_DIR, "img")
LABEL_DIR = os.path.join(DATA_DIR, "label")

IM_DIR_TRIAL = os.path.join(IM_DIR, "trial")
IM_DIR_TEST = os.path.join(IM_DIR, "test")
LABEL_DIR_TRIAL = os.path.join(LABEL_DIR, "trial")

In [12]:
# for i in range(1,5):
#     if not os.path.exists(os.path.join(IM_DIR_TRIAL, str(i))):
#         os.mkdir(os.path.join(IM_DIR_TRIAL, str(i)))

# with open(os.path.join(DATA_DIR, "label.json"), 'r') as f:
#     labels = json.load(f)

# for im_name in os.listdir(IM_DIR_TRIAL):
#     if im_name[8:-4] not in labels:
#         continue
#     risk_level = labels[im_name[8:-4]]
#     shutil.move(os.path.join(IM_DIR_TRIAL, im_name), os.path.join(IM_DIR_TRIAL, str(risk_level)))

In [13]:
im_data = tf.keras.utils.image_dataset_from_directory(IM_DIR_TRIAL, labels="inferred", batch_size=32, image_size=(224,224))

Found 1592 files belonging to 4 classes.


I0000 00:00:1779309400.169002  224357 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13419 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Ti, pci bus id: 0000:01:00.0, compute capability: 12.0


In [7]:
# 1. Load the data (High-level API)
IMG_SIZE = (224,224)
BATCH_SIZE = 32

train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
    IM_DIR_TRIAL,
    validation_split=0.2,
    subset="both",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Optimize performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 2. Build your CNN model
# Note: High-level API loads images with pixels from 0 to 255. 
# We include a Rescaling layer to normalize them to [0, 1].
model = tf.keras.models.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(224, 224, 3)), 
    
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(2, activation='softmax') # Use 'softmax' and class_number for multi-class
])

# 3. Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # Use 'sparse_categorical_crossentropy' for multi-class
    metrics=['accuracy']
)

# 4. Feed the data
# Keras automatically detects the batch sizes and steps per epoch from the tf.data object
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)

Found 704 files belonging to 2 classes.
Using 564 files for training.
Using 140 files for validation.


/home/unb/miniforge3/envs/tf_cuda/lib/python3.11/site-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20


I0000 00:00:1779381892.224874  290772 service.cc:152] XLA service 0x7de704003a00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779381892.224890  290772 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 5060 Ti, Compute Capability 12.0
2026-05-21 13:44:52.236698: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779381892.328154  290772 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-05-21 13:44:52.431692: W external/local_xla/xla/backends/gpu/codegen/triton/fusion_emitter.cc:1240] Triton does not support sm_120 yet. Passing CC 10.0 to avoid spurious "unsupported conversion" errors
2026-05-21 13:44:52.431703: W external/local_xla/xla/backends/gpu/codegen/triton/fusion_emitter.cc:1240] Triton does not support sm_120 yet. Passing CC 10.0 to avoid spurious "unsupported conversion" errors
2026-05-21 13:4

 7/18 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.6200 - loss: 0.6445

...

18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.7204 - loss: 0.5534

2026-05-21 13:44:58.210917: W external/local_xla/xla/backends/gpu/codegen/triton/fusion_emitter.cc:1240] Triton does not support sm_120 yet. Passing CC 10.0 to avoid spurious "unsupported conversion" errors
2026-05-21 13:44:58.212064: W external/local_xla/xla/backends/gpu/codegen/triton/fusion_emitter.cc:1240] Triton does not support sm_120 yet. Passing CC 10.0 to avoid spurious "unsupported conversion" errors
2026-05-21 13:44:58.213572: W external/local_xla/xla/backends/gpu/codegen/triton/fusion_emitter.cc:1240] Triton does not support sm_120 yet. Passing CC 10.0 to avoid spurious "unsupported conversion" errors
2026-05-21 13:44:58.213660: W external/local_xla/xla/backends/gpu/codegen/triton/fusion_emitter.cc:1240] Triton does not support sm_120 yet. Passing CC 10.0 to avoid spurious "unsupported conversion" errors
2026-05-21 13:44:58.213769: W external/local_xla/xla/backends/gpu/codegen/triton/fusion_emitter.cc:1240] Triton does not support sm_120 yet. Passing CC 10.0 to avoid spurio

18/18 ━━━━━━━━━━━━━━━━━━━━ 8s 283ms/step - accuracy: 0.8209 - loss: 0.4324 - val_accuracy: 0.9143 - val_loss: 0.2136
Epoch 2/20
 1/18 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.8438 - loss: 0.3558

2026-05-21 13:45:00.051215: W external/local_xla/xla/backends/gpu/codegen/triton/fusion_emitter.cc:1240] Triton does not support sm_120 yet. Passing CC 10.0 to avoid spurious "unsupported conversion" errors
2026-05-21 13:45:00.066821: W external/local_xla/xla/backends/gpu/codegen/triton/fusion_emitter.cc:1240] Triton does not support sm_120 yet. Passing CC 10.0 to avoid spurious "unsupported conversion" errors
2026-05-21 13:45:00.076085: W external/local_xla/xla/backends/gpu/codegen/triton/fusion_emitter.cc:1240] Triton does not support sm_120 yet. Passing CC 10.0 to avoid spurious "unsupported conversion" errors


18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9060 - loss: 0.3048 - val_accuracy: 0.9643 - val_loss: 0.2389
Epoch 3/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9255 - loss: 0.1710 - val_accuracy: 0.9214 - val_loss: 0.1156
Epoch 4/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9397 - loss: 0.1306 - val_accuracy: 0.9429 - val_loss: 0.0991
Epoch 5/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9734 - loss: 0.0957 - val_accuracy: 0.9500 - val_loss: 0.0870
Epoch 6/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9716 - loss: 0.0770 - val_accuracy: 0.9929 - val_loss: 0.0303
Epoch 7/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9840 - loss: 0.0486 - val_accuracy: 0.9929 - val_loss: 0.0268
Epoch 8/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9911 - loss: 0.0422 - val_accuracy: 0.9929 - val_loss: 0.0249
Epoch 9/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9929 - loss: 0.0325 - val_accuracy: 0.9929 - val_loss: 0.

In [31]:
test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(IM_DIR_TEST),
    image_size=IMG_SIZE,
    batch_size=1,
    shuffle=False
)

def normalize_image(image, label):
    return tf.cast(image / 255.0, tf.float32), label

# def filter_label(image, label):
#     return label == 

# test_ds_2 = filter(, test_ds)
    
test_ds = test_ds.map(normalize_image)

AUTOTUNE = tf.data.AUTOTUNE
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

y_pred = model.predict(test_ds)

Found 888 files belonging to 2 classes.
888/888 ━━━━━━━━━━━━━━━━━━━━ 1s 849us/step


In [43]:
n_level_2 = len(os.listdir(os.path.join(IM_DIR_TEST, "2")))
print(y_pred[:n_level_2,0].mean())
print(y_pred[n_level_2:,0].mean())

0.8678273
0.8720024


In [ ]:
# 2. Convert probabilities to class indices
y_preds = tf.argmax(y_probs, axis=1)

# 3. Create the confusion matrix
cm = tf.math.confusion_matrix(labels=test_labels, predictions=y_preds)